# Part 2: Báo Cáo Biến Đổi Dữ Liệu & Kiểm Định Hợp Đồng (Data Contract)
**Thành viên phụ trách:** Thành viên 2 | **Nhánh Git:** `feature/transformation`
---
Notebook kiểm định việc tính toán thời gian, xử lý qua nửa đêm, gán nhãn 3 lớp và kiểm tra 8 ràng buộc schema.

### 1. Kiểm Tra Tự Động 8 Ràng Buộc Của Data Contract


In [ ]:
from pipeline.check_data import validate_flights_data
from config import FILE_CLEANED
import pandas as pd

passed, checks, metrics = validate_flights_data(FILE_CLEANED)
display(pd.DataFrame(checks))
print(f'Toàn bộ kiểm định đạt chuẩn: {passed}')
print(f'Tổng số dòng: {metrics["total_rows"]:,}')


### 2. Kiểm Tra Thuật Toán Xử Lý Thời Gian Qua Nửa Đêm


In [ ]:
from pipeline.transformation import calculate_delay

# Test 1: Lịch bay 23:50, thực tế 00:20 hôm sau -> trễ 30 phút (+30)
d1 = calculate_delay('23:50', None, '00:20', '')
# Test 2: Lịch bay 00:15, cất cánh sớm 23:55 hôm trước -> bay sớm 20 phút (-20)
d2 = calculate_delay('00:15', None, '23:55', '')
print(f'Test 1 (23:50 -> 00:20): delay = {d1} phút (Kỳ vọng: 30.0)')
print(f'Test 2 (00:15 -> 23:55): delay = {d2} phút (Kỳ vọng: -20.0)')
assert d1 == 30.0 and d2 == -20.0
print('Thuật toán tính độ trễ nửa đêm hoàn toàn chính xác!')


### 3. Phân Bố Nhãn Mục Tiêu (status_clean) Theo Ngưỡng 15 Phút


In [ ]:
import matplotlib.pyplot as plt
df = pd.read_csv(FILE_CLEANED)
vc = df['status_clean'].value_counts()
plt.figure(figsize=(7, 4))
colors = ['#10b981', '#f59e0b', '#ef4444']
vc.plot(kind='bar', color=colors, edgecolor='black')
plt.title('Phân bố Nhãn Mục Tiêu (Target: status_clean)')
plt.ylabel('Số lượng chuyến bay')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(vc.to_frame('Số lượng'))
